
### Fase 6: Modelagem de Tópicos (Topic Modeling)

Nesta fase, vamos pegar apenas as reclamações classificadas como NEGATIVAS e usar um algoritmo não-supervisionado (BERTopic) para agrupar automaticamente essas reclamações em temas (ex: "Cobrança Indevida", "Cartão Bloqueado", "Juros Altos").

Isso responderá à pergunta do Tech Challenge: "Quais são as principais dores dos clientes por produto?"

🛠️ Pré-requisitos O BERTopic é a ferramenta State-of-the-Art para isso. Ele usa embeddings (como o RoBERTa) + Clusterização (HDBSCAN).

💻 Código: Fase 6 (Análise de Dores) Este script:

Carrega o processed_dataset.parquet.

Filtra apenas as reclamações Negativas (label_final == 0).

Executa o BERTopic para descobrir os 10 principais problemas.

Gera um gráfico HTML interativo e uma tabela estática.



[Voltar para notebook principal](./0_notebook_principal.ipynb)

In [1]:
import polars as pl
from bertopic import BERTopic
from pathlib import Path
import os

# --- Configurações ---
BASE_DIR = Path.cwd()
INPUT_FILE = BASE_DIR / "data" / "parquet" / "processed_dataset.parquet"
OUTPUT_DIR = BASE_DIR / "reports" / "topics"

def analyze_pain_points():
    print("--- 📊 FASE 6: Análise de Dores (Topic Modeling) ---")
    
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    # 1. Carregar Dados e Filtrar Negativos
    print("📂 Carregando dataset processado...")
    df = pl.read_parquet(INPUT_FILE)
    
    # Filtramos apenas as reclamações NEGATIVAS (Label 0)
    # Pois queremos analisar as "Dores" e não os elogios.
    neg_df = df.filter(pl.col("label_final") == 0)
    
    print(f"   Total de reclamações negativas para análise: {len(neg_df):,}")
    
    # Para o BERTopic não demorar horas, vamos usar uma amostra de 10k-20k
    # Se tiver GPU forte, pode aumentar para 50k.
    if len(neg_df) > 20000:
        print("⚠️  Amostrando 20.000 reclamações para agilizar a clusterização...")
        neg_df = neg_df.sample(20000, seed=42)
    
    # Usamos o texto LIMPO (cleaned_text) para os tópicos ficarem mais bonitos (sem stopwords)
    docs = neg_df["cleaned_text"].to_list()

    # 2. Inicializar BERTopic
    print("🚀 Iniciando BERTopic (Isso usa GPU para embeddings)...")
    
    # min_topic_size: Tamanho mínimo para considerar um grupo um "tópico"
    topic_model = BERTopic(language="english", calculate_probabilities=False, min_topic_size=50, verbose=True)
    
    # 3. Fit & Transform
    topics, probs = topic_model.fit_transform(docs)
    
    # 4. Resultados
    print("\n🏆 Top 10 Tópicos de Reclamação:")
    freq = topic_model.get_topic_info()
    print(freq.head(11)) # O tópico -1 é "ruído" (outliers), ignorar

    # 5. Salvar Visualizações
    print(f"\n💾 Salvando visualizações em: {OUTPUT_DIR}")
    
    # Gráfico de Barras (Palavras-chave por tópico)
    fig_bar = topic_model.visualize_barchart(top_n_topics=10)
    fig_bar.write_html(OUTPUT_DIR / "top_pain_points.html")
    
    # Hierarquia (Como os problemas se relacionam)
    fig_hierarchy = topic_model.visualize_hierarchy(top_n_topics=20)
    fig_hierarchy.write_html(OUTPUT_DIR / "topic_hierarchy.html")
    
    # Salvar a tabela de tópicos em CSV para seu relatório
    freq.to_csv(OUTPUT_DIR / "topic_frequency.csv", index=False)
    
    print("✅ Análise concluída! Abra o arquivo 'top_pain_points.html' no navegador.")

if __name__ == "__main__":
    analyze_pain_points()

--- 📊 FASE 6: Análise de Dores (Topic Modeling) ---
📂 Carregando dataset processado...
   Total de reclamações negativas para análise: 479,242
⚠️  Amostrando 20.000 reclamações para agilizar a clusterização...
🚀 Iniciando BERTopic (Isso usa GPU para embeddings)...


2026-03-15 08:19:43,055 - BERTopic - Embedding - Transforming documents to embeddings.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/625 [00:00<?, ?it/s]

2026-03-15 08:20:12,875 - BERTopic - Embedding - Completed ✓
2026-03-15 08:20:12,876 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-03-15 08:21:08,505 - BERTopic - Dimensionality - Completed ✓
2026-03-15 08:21:08,507 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-03-15 08:21:23,279 - BERTopic - Cluster - Completed ✓
2026-03-15 08:21:23,415 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-03-15 08:21:24,588 - BERTopic - Representation - Completed ✓



🏆 Top 10 Tópicos de Reclamação:
    Topic  Count                                   Name  \
0      -1   7314             -1_xxxx_credit_not_account   
1       0    761       0_theft_identity_xxxx_fraudulent   
2       1    708           1_cash_app_resolution_unfair   
3       2    585         2_debt_collection_call_company   
4       3    477                3_card_bank_money_close   
5       4    406   4_equifax_xxxx_investigation_dispute   
6       5    367                 5_xxxx_xx_balance_fcra   
7       6    269  6_inquiry_hard_unauthorized_authorize   
8       7    248             7_navy_fee_transaction_lag   
9       8    217     8_contain_error_inaccuracy_mistake   
10      9    206          9_late_payment_company_remark   

                                       Representation  \
0   [xxxx, credit, not, account, report, xx, infor...   
1   [theft, identity, xxxx, fraudulent, victim, ft...   
2   [cash, app, resolution, unfair, efta, unprotec...   
3   [debt, collection, call, c

In [1]:
import polars as pl
import matplotlib.pyplot as plt
import seaborn as sns
from bertopic import BERTopic
from pathlib import Path
import os

# --- Configurações ---
BASE_DIR = Path.cwd()
PROCESSED_FILE = BASE_DIR / "data" / "parquet" / "processed_dataset.parquet"
SILVER_FILE = BASE_DIR / "data" / "parquet" / "complaints_lean_2025.parquet"
OUTPUT_DIR = BASE_DIR / "reports" / "business_insights"

def run_business_intelligence():
    print("--- 📊 FASE 6: Business Intelligence & Análise de Dores ---")
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    # 1. Carregar Dados e fazer o JOIN
    print("📂 Carregando dados e resgatando metadados (Produto, Empresa, etc)...")
    
    df_processed = pl.read_parquet(PROCESSED_FILE)
    
    # Carregamos apenas as colunas úteis da camada Silver
    df_silver = pl.read_parquet(SILVER_FILE).select([
        "complaint_id", "product", "sub_product", "issue", "company"
    ])
    
    # Unimos os datasets pelo ID
    df_full = df_processed.join(df_silver, on="complaint_id", how="inner")
    
    # Filtramos apenas as RECLAMAÇÕES NEGATIVAS (label_final == 0)
    df_neg = df_full.filter(pl.col("label_final") == 0)
    print(f"   Total de reclamações negativas para análise: {len(df_neg):,}")

    # =====================================================================
    # PARTE A: ANÁLISE DESCRITIVA (BUSINESS INSIGHTS)
    # =====================================================================
    print("\n📈 Gerando Gráficos de Negócio...")
    
    # Configuração visual do Seaborn
    sns.set_theme(style="whitegrid")

    # Insight 1: Top 10 Produtos Mais Reclamados
    top_products = df_neg.group_by("product").len().sort("len", descending=True).head(10).to_pandas()
    plt.figure(figsize=(10, 6))
    sns.barplot(data=top_products, y="product", x="len", palette="Reds_r")
    plt.title("Top 10 Produtos com Mais Reclamações Negativas")
    plt.xlabel("Volume de Reclamações")
    plt.ylabel("Produto")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "top_products.png")
    plt.close()

    # Insight 2: Top 10 Problemas Originais (Classificação do Banco)
    top_issues = df_neg.group_by("issue").len().sort("len", descending=True).head(10).to_pandas()
    plt.figure(figsize=(10, 6))
    sns.barplot(data=top_issues, y="issue", x="len", palette="Oranges_r")
    plt.title("Top 10 Motivos de Reclamação (Classificação Padrão)")
    plt.xlabel("Volume de Reclamações")
    plt.ylabel("Motivo (Issue)")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "top_issues.png")
    plt.close()

    # Insight 3: Top 10 Empresas Mais Reclamadas (Caso seu dataset tenha várias)
    top_companies = df_neg.group_by("company").len().sort("len", descending=True).head(10).to_pandas()
    plt.figure(figsize=(10, 6))
    sns.barplot(data=top_companies, y="company", x="len", palette="Purples_r")
    plt.title("Top 10 Empresas com Mais Reclamações")
    plt.xlabel("Volume")
    plt.ylabel("Empresa")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "top_companies.png")
    plt.close()
    
    print("✅ Gráficos de Negócio salvos na pasta 'reports/business_insights/'")

    # =====================================================================
    # PARTE B: TOPIC MODELING COM BERTOPIC (A Dificuldade Oculta)
    # =====================================================================
    print("\n🚀 Iniciando BERTopic para descobrir problemas não mapeados...")
    
    # Amostragem para evitar tempo excessivo de execução (ajuste conforme sua máquina)
    sample_size = min(20000, len(df_neg))
    df_sample = df_neg.sample(sample_size, seed=42)
    docs = df_sample["cleaned_text"].to_list()

    # Inicializar BERTopic
    topic_model = BERTopic(language="english", calculate_probabilities=False, min_topic_size=50)
    
    # Fit & Transform
    topics, probs = topic_model.fit_transform(docs)
    
    # Salvar Resultados do BERTopic
    freq = topic_model.get_topic_info()
    freq.to_csv(OUTPUT_DIR / "bertopic_frequency.csv", index=False)
    
    # Gráfico de Barras dos Tópicos Encontrados
    fig_bar = topic_model.visualize_barchart(top_n_topics=10)
    fig_bar.write_html(OUTPUT_DIR / "top_pain_points_bertopic.html")
    
    print("✅ Análise BERTopic concluída! Arquivo HTML e CSV salvos.")
    print("\n🎉 FASE 6 FINALIZADA COM SUCESSO!")

if __name__ == "__main__":
    run_business_intelligence()

--- 📊 FASE 6: Business Intelligence & Análise de Dores ---
📂 Carregando dados e resgatando metadados (Produto, Empresa, etc)...
   Total de reclamações negativas para análise: 479,242

📈 Gerando Gráficos de Negócio...
✅ Gráficos de Negócio salvos na pasta 'reports/business_insights/'

🚀 Iniciando BERTopic para descobrir problemas não mapeados...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Análise BERTopic concluída! Arquivo HTML e CSV salvos.

🎉 FASE 6 FINALIZADA COM SUCESSO!


### Top 10 Produtos Mais Reclamados
![Top Produtos](reports/business_insights/top_products.png)

### Top 10 Motivos de Reclamação
![Top Motivos](reports/business_insights/top_issues.png)

### Top 10 Empresas com Mais Reclamações
![Top Motivos](reports/business_insights/top_companies.png)


[Voltar para notebook principal](./0_notebook_principal.ipynb)